<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/LLM/LLM04/260929_Private_RAG_pjt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 나의 문서로 로컬 RAG 만들고 개선하기

## 사내 복무 규정 기반 Local RAG + Hybrid Search

실험 흐름: 문서 준비 → 구조 기반 청킹 → BGE-M3 Baseline → Qdrant → 로컬 LLM → 오프라인 검증 → BM25+RRF → RRF k 실험 → 조용한 실패 3종 진단(Track 3)

In [1]:
!pip install -q "langchain==0.3.*" "langchain-community==0.3.*" "langchain-text-splitters==0.3.*" "langchain-experimental==0.3.*" "langchain-huggingface==0.3.*"
!pip install -q "sentence-transformers==5.7.*" "qdrant-client==1.12.*"
!pip install -q "transformers==4.57.*" "accelerate==1.14.*" "bitsandbytes==0.50.*" "rank_bm25==0.2.2"

import os, re
import numpy as np
import pandas as pd
import torch
import transformers, sentence_transformers

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"transformers {transformers.__version__} / sentence-transformers {sentence_transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is inc

In [2]:
%%writefile company_policy.md
# (주)아이펠테크 임직원 복무 규정

## 제1장 총칙

### 제1조 (목적)
본 규정은 (주)아이펠테크(이하 "회사") 임직원의 복무에 관한 기본 사항을 정함을 목적으로 한다. 본 규정에서 정하지 아니한 사항은 취업규칙 및 관계 법령에 따른다.

### 제2조 (적용 범위)
본 규정은 회사에 재직 중인 모든 정규직 및 계약직 임직원에게 적용된다. 파견 근로자의 복무에 관한 사항은 파견 계약에서 별도로 정한다.

## 제2장 근무

### 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 한다. 시업 시각은 오전 10시, 종업 시각은 오후 7시로 하되, 유연근무제 적용자는 제4조에 따른다.

### 제4조 (유연근무제)
임직원은 오전 8시부터 오전 11시 사이에 시업 시각을 자율적으로 선택할 수 있다. 유연근무제를 적용받으려는 임직원은 매월 말일까지 다음 달의 시업 시각을 인사 시스템에 등록하여야 한다. 코어 타임은 오전 11시부터 오후 4시까지로 하며, 코어 타임 중에는 근무 장소에서 업무에 임하여야 한다.

### 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다. 다만 부서 필수 회의가 있는 날에는 부서장이 재택근무를 제한할 수 있다. 재택근무 시에도 코어 타임 규정은 동일하게 적용된다.

## 제3장 휴가

### 제6조 (연차 유급 휴가)
회사는 근로기준법에 따라 1년간 80% 이상 출근한 임직원에게 15일의 연차 유급 휴가를 부여한다. 3년 이상 계속 근로한 임직원에게는 최초 1년을 초과하는 계속 근로 연수 매 2년마다 1일을 가산한 연차 휴가를 부여하며, 가산 휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.

### 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

### 제8조 (경조 휴가)
임직원은 다음 각 호의 사유가 발생한 경우 경조 휴가를 신청할 수 있다. 본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 본인 및 배우자의 부모 사망 5일, 조부모 및 형제자매 사망 3일. 경조 휴가는 사유 발생일로부터 30일 이내에 사용하여야 한다.

## 제4장 보안

### 제9조 (정보 보안 서약)
임직원은 입사 시 정보 보안 서약서를 제출하여야 하며, 재직 중 취득한 회사의 영업 비밀 및 고객 정보를 외부에 유출하여서는 아니 된다. 이는 퇴직 후 2년간 동일하게 적용된다.

### 제10조 (사내 데이터의 외부 반출)
고객 개인정보, 미공개 재무 정보, 기술 설계 문서를 포함한 사내 데이터는 정보보호팀장의 사전 승인 없이 외부 클라우드 서비스, 외부 API, 개인 저장 장치로 반출할 수 없다. 외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.

### 제11조 (위반 시 조치)
제9조 및 제10조를 위반한 임직원은 상벌 규정에 따라 징계할 수 있으며, 회사에 손해가 발생한 경우 배상을 청구할 수 있다.

Writing company_policy.md


In [3]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

with open("company_policy.md", encoding="utf-8") as f:
    document = f.read()

fixed_chunks = CharacterTextSplitter(
    separator="", chunk_size=300, chunk_overlap=50
).split_text(document)

recursive_chunks = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
).split_text(document)

md_splitter = MarkdownHeaderTextSplitter([
    ("#", "문서"),
    ("##", "장"),
    ("###", "조항"),
])
md_chunks = md_splitter.split_text(document)

strategies = {
    "고정 크기": fixed_chunks,
    "재귀적": recursive_chunks,
    "구조 기반": [c.page_content for c in md_chunks],
}

print(f"문서 길이 : {len(document)}자")
for name, items in strategies.items():
    lengths = [len(x) for x in items]
    print(f"{name}: {len(items)}개 / 평균 {sum(lengths)/len(lengths):.0f}자")

chunks = md_chunks
chunk_texts = [c.page_content for c in chunks]

for i, c in enumerate(chunks):
    print(f"[{i}] {c.metadata.get('조항', '')}")

문서 길이 : 1598자
고정 크기: 7개 / 평균 271자
재귀적: 8개 / 평균 201자
구조 기반: 11개 / 평균 120자
[0] 제1조 (목적)
[1] 제2조 (적용 범위)
[2] 제3조 (근무 시간)
[3] 제4조 (유연근무제)
[4] 제5조 (재택근무)
[5] 제6조 (연차 유급 휴가)
[6] 제7조 (연차 휴가의 이월)
[7] 제8조 (경조 휴가)
[8] 제9조 (정보 보안 서약)
[9] 제10조 (사내 데이터의 외부 반출)
[10] 제11조 (위반 시 조치)


## 평가 세트

총 15문항으로 구성한다. 의미 질문 11개와 식별자 질문 4개를 포함한다.
Gold index는 Markdown 구조 기반 청킹 결과에서 `제1조=0`부터 순서대로 확인한다.

In [4]:
eval_set = [
    ("회사 규정의 목적은 무엇인가요?", 0),
    ("이 규정은 어떤 직원에게 적용되나요?", 1),
    ("하루 근무 시간은 몇 시간인가요?", 2),
    ("출근 시간을 자유롭게 정할 수 있나요?", 3),
    ("재택근무는 일주일에 몇 번까지 가능한가요?", 4),
    ("입사 후 연차 휴가는 며칠인가요?", 5),
    ("사용하지 않은 연차를 다음 해로 넘길 수 있나요?", 6),
    ("배우자가 출산하면 휴가를 며칠 받을 수 있나요?", 7),
    ("퇴직한 뒤에도 정보 보안 의무가 적용되나요?", 8),
    ("회사 문서를 외부 AI 서비스에 입력해도 되나요?", 9),
    ("제11조에는 어떤 조치가 규정되어 있나요?", 10),
]

id_eval_set = [
    ("제7조가 무슨 내용이야?", 6),
    ("제10조 알려줘", 9),
    ("제4조 규정 보여줘", 3),
    ("제8조에 뭐라고 써있어?", 7),
]

combined_eval = eval_set + id_eval_set
print("의미/규정 질문 :", len(eval_set))
print("식별자 질문    :", len(id_eval_set))
print("전체            :", len(combined_eval))

의미/규정 질문 : 11
식별자 질문    : 4
전체            : 15


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

tok_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tok_bge = AutoTokenizer.from_pretrained("BAAI/bge-m3")

bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

e5_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

bge_doc_vecs = np.array(bge_embeddings.embed_documents(chunk_texts))
e5_doc_vecs = np.array(
    e5_embeddings.embed_documents([f"passage: {t}" for t in chunk_texts])
)

def hit_at_k(query_vec, doc_vecs, gold_idx, k=3):
    scores = doc_vecs @ query_vec
    top_k = np.argsort(scores)[::-1][:k]
    return int(gold_idx in top_k), top_k

def evaluate_dense(name, embeddings, doc_vecs, dataset, query_prefix=""):
    hits1 = hits3 = 0
    for q, gold in dataset:
        q_vec = np.array(embeddings.embed_query(query_prefix + q))
        h1, _ = hit_at_k(q_vec, doc_vecs, gold, 1)
        h3, top3 = hit_at_k(q_vec, doc_vecs, gold, 3)
        hits1 += h1
        hits3 += h3
        print(("O" if h3 else "X"), q, "->", list(top3), "gold:", gold)
    n = len(dataset)
    return hits1/n, hits3/n

bge_scores = evaluate_dense(
    "BGE-M3", bge_embeddings, bge_doc_vecs, eval_set
)
e5_scores = evaluate_dense(
    "E5", e5_embeddings, e5_doc_vecs, eval_set, query_prefix="query: "
)

e5_doc_vecs_noprefix = np.array(
    e5_embeddings.embed_documents(chunk_texts)
)
e5_no_prefix_scores = evaluate_dense(
    "E5 prefix 없음",
    e5_embeddings,
    e5_doc_vecs_noprefix,
    eval_set,
)

print(f"BGE-M3 Hit@3 : {bge_scores[1]:.1%}")
print(f"E5 Hit@3     : {e5_scores[1]:.1%}")
print(f"E5 no-prefix : {e5_no_prefix_scores[1]:.1%}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

O 회사 규정의 목적은 무엇인가요? -> [np.int64(0), np.int64(1), np.int64(8)] gold: 0
O 이 규정은 어떤 직원에게 적용되나요? -> [np.int64(1), np.int64(0), np.int64(8)] gold: 1
O 하루 근무 시간은 몇 시간인가요? -> [np.int64(2), np.int64(3), np.int64(4)] gold: 2
O 출근 시간을 자유롭게 정할 수 있나요? -> [np.int64(3), np.int64(2), np.int64(4)] gold: 3
O 재택근무는 일주일에 몇 번까지 가능한가요? -> [np.int64(4), np.int64(5), np.int64(2)] gold: 4
O 입사 후 연차 휴가는 며칠인가요? -> [np.int64(5), np.int64(6), np.int64(2)] gold: 5
O 사용하지 않은 연차를 다음 해로 넘길 수 있나요? -> [np.int64(6), np.int64(5), np.int64(7)] gold: 6
O 배우자가 출산하면 휴가를 며칠 받을 수 있나요? -> [np.int64(5), np.int64(7), np.int64(6)] gold: 7
O 퇴직한 뒤에도 정보 보안 의무가 적용되나요? -> [np.int64(8), np.int64(9), np.int64(1)] gold: 8
O 회사 문서를 외부 AI 서비스에 입력해도 되나요? -> [np.int64(9), np.int64(8), np.int64(3)] gold: 9
O 제11조에는 어떤 조치가 규정되어 있나요? -> [np.int64(10), np.int64(0), np.int64(1)] gold: 10
O 회사 규정의 목적은 무엇인가요? -> [np.int64(0), np.int64(1), np.int64(8)] gold: 0
O 이 규정은 어떤 직원에게 적용되나요? -> [np.int64(1), np.int64(0), np.int64(8)] gold: 1
O 하루 근무 시간은 몇 시

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(path="./qdrant_local")
COLLECTION = "company_policy"

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=bge_doc_vecs[i].tolist(),
        payload={
            "text": chunks[i].page_content,
            "장": chunks[i].metadata.get("장", ""),
            "조항": chunks[i].metadata.get("조항", ""),
        },
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION, points=points)

def qdrant_rank(query, top_k=3):
    q_vec = bge_embeddings.embed_query(query)
    results = client.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        limit=top_k,
    ).points
    return [int(r.id) for r in results]

def hit3(rank_fn, dataset):
    return sum(gold in rank_fn(q, 3) for q, gold in dataset) / len(dataset)

baseline_hit3 = hit3(qdrant_rank, combined_eval)
print(f"Baseline Hit@3 (전체 15문항) : {baseline_hit3:.1%}")

Baseline Hit@3 (전체 15문항) : 73.3%


## 로컬 LLM + RAG
Qwen2.5-7B-Instruct를 4bit로 로드하고 Qdrant 검색 결과만 근거로 답변하도록 구성한다.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model.eval()

def generate(system_prompt, user_prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

SYSTEM_PROMPT = """당신은 (주)아이펠테크의 사내 규정 안내 챗봇입니다.
1. 제공된 규정 내용에 근거해서만 답변합니다.
2. 답변 끝에 근거 조항을 표기합니다.
3. 규정에 없는 내용이면 "규정에서 확인되지 않습니다"라고 답합니다.
4. 답변은 반드시 한국어로 작성합니다."""

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [17]:
def retrieve(query, top_k=3):
    q_vec = bge_embeddings.embed_query(query)
    results = client.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        limit=top_k,
    ).points

    return [
        {
            "조항": r.payload["조항"],
            "내용": r.payload["text"],
            "유사도": r.score,
        }
        for r in results
    ]

def rag_answer(query, top_k=3):
    contexts = retrieve(query, top_k)
    context_text = "\n\n".join(
        f"[{c['조항']}]\n{c['내용']}" for c in contexts
    )
    prompt = (
        "다음은 사내 규정에서 검색된 내용입니다.\n\n"
        f"{context_text}\n\n질문 : {query}"
    )
    return generate(SYSTEM_PROMPT, prompt)

def rag_answer_v2(query, top_k=3, threshold=0.4):
    contexts = retrieve(query, top_k)
    if not contexts or contexts[0]["유사도"] < threshold:
        return "규정에서 확인되지 않습니다."
    context_text = "\n\n".join(
        f"[{c['조항']}]\n{c['내용']}" for c in contexts
    )
    prompt = (
        "다음은 사내 규정에서 검색된 내용입니다.\n\n"
        f"{context_text}\n\n질문 : {query}"
    )
    return generate(SYSTEM_PROMPT, prompt)

print(rag_answer("연차를 다음 해로 넘겨서 쓸 수 있어?"))
print("*"*90)
print(rag_answer_v2("점심시간은 몇 시야?"))

네, 회사의 업무상 사유로 당해 연도에 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.

- 근거: 제7조 (연차 휴가의 이월)
******************************************************************************************
사내 규정에서는 점심시간에 대한 구체적인 시간이 명시되어 있지 않습니다. 따라서 점심시간은 기본 근무 시간 8시간에서 시업과 종업 시간을 뺀 나머지 시간에 포함되어 있다고 볼 수 있습니다. 

따라서, 기본 근무 시간은 오전 10시부터 오후 7시까지이며, 유연근무제를 적용받는 임직원의 시업 시간은 오전 8시부터 오전 11시 사이로 선택할 수 있으므로, 점심시간은 각 개인의 선택에 따라 다르게 될 것입니다.

규정에서 확인되지 않습니다.


In [9]:
# 폐쇄 환경 검증
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(rag_answer("배우자가 출산하면 휴가 며칠이야?"))
print("오프라인 모드 실행 완료")

배우자가 출산하면 경조 휴가를 10일 사용할 수 있습니다. 

- 귀하의 질문에 대한 근거 조항: [제8조 (경조 휴가)]
오프라인 모드 실행 완료


## Track 2 — Hybrid Search (BM25 + RRF)
식별자와 키워드 일치에 강한 BM25와 의미 검색에 강한 Dense Retrieval을 RRF로 결합한다.

In [18]:
from rank_bm25 import BM25Okapi

full_texts = [
    f"{c.metadata.get('장', '')} {c.metadata.get('조항', '')}\n{c.page_content}"
    for c in chunks
]

def tokenize_ko(text):
    words = re.findall(r"[가-힣]+|[A-Za-z]+|[0-9]+", text)
    identifiers = re.findall(r"제\d+조|제\d+항", text)
    return words + identifiers

bm25 = BM25Okapi([tokenize_ko(x) for x in full_texts])
hybrid_dense_vecs = np.array(
    bge_embeddings.embed_documents(full_texts)
)

def dense_rank(query, top_n=10):
    q_vec = np.array(bge_embeddings.embed_query(query))
    scores = hybrid_dense_vecs @ q_vec
    return list(np.argsort(scores)[::-1][:top_n])

def bm25_rank(query, top_n=10):
    scores = bm25.get_scores(tokenize_ko(query))
    return list(np.argsort(scores)[::-1][:top_n])

def rrf(rank_lists, k=60, top_n=3):
    scores = {}
    for ranks in rank_lists:
        for position, idx in enumerate(ranks, start=1):
            scores[idx] = scores.get(idx, 0) + 1 / (k + position)
    return sorted(scores, key=scores.get, reverse=True)[:top_n]

def hybrid_rank(query, top_n=3, rrf_k=60):
    return rrf(
        [dense_rank(query, 10), bm25_rank(query, 10)],
        k=rrf_k,
        top_n=top_n,
    )

combined_eval = eval_set + id_eval_set

hybrid_k60 = hit3(
    lambda q, k=3: hybrid_rank(q, k, 60),
    combined_eval,
)

print(f"Hybrid RRF(60) Hit@3 (전체 15문항) : {hybrid_k60:.1%}")

Hybrid RRF(60) Hit@3 (전체 15문항) : 100.0%


In [11]:
rrf_rows = []

for k_value in [10, 60, 100, 200]:
    score = hit3(
        lambda q, top_k=3, kk=k_value: hybrid_rank(q, top_k, kk),
        combined_eval,
    )
    rrf_rows.append({
        "RRF k": k_value,
        "Hit@3": score,
        "Baseline 대비": score - baseline_hit3,
    })

rrf_df = pd.DataFrame(rrf_rows)
display(
    rrf_df.assign(
        **{
            "Hit@3": rrf_df["Hit@3"].map(lambda x: f"{x:.1%}"),
            "Baseline 대비": rrf_df["Baseline 대비"].map(lambda x: f"{x:+.1%}"),
        }
    )
)

,RRF k,Hit@3,Baseline 대비
0,10,100.0%,+26.7%
1,60,100.0%,+26.7%
2,100,100.0%,+26.7%
3,200,100.0%,+26.7%


## Track 3 — 조용한 실패 3종 진단
성능 개선과 별도로 토큰 절단, E5 prefix, 정규화 문제를 진단한다.

In [12]:
# 1) Token truncation
items = []
for i in range(1, 60):
    items.append(
        f"{i}. 세부 시행 절차는 담당 부서의 내부 지침에 따르며 "
        f"관련 부서와 협의하여 조정할 수 있다."
    )

long_clause = (
    "제12조 (세부 시행 절차)\n"
    + "\n".join(items)
    + "\n60. 재해 발생 시 비상연락망은 총무팀장이 관리하며 매 분기 갱신한다."
)

print("E5 토큰 :", len(tok_e5.encode(long_clause)))
print("BGE 토큰:", len(tok_bge.encode(long_clause)))

test_corpus = chunk_texts + [long_clause]
gold_idx = len(test_corpus) - 1
query = "비상연락망은 누가 관리해?"

def rank_of(vecs, q_vec, target):
    scores = vecs @ q_vec
    order = list(np.argsort(scores)[::-1])
    return order.index(target) + 1

e5_test_vecs = np.array(
    e5_embeddings.embed_documents([f"passage: {x}" for x in test_corpus])
)
e5_rank = rank_of(
    e5_test_vecs,
    np.array(e5_embeddings.embed_query(f"query: {query}")),
    gold_idx,
)

bge_test_vecs = np.array(bge_embeddings.embed_documents(test_corpus))
bge_rank = rank_of(
    bge_test_vecs,
    np.array(bge_embeddings.embed_query(query)),
    gold_idx,
)

print(f"E5  긴 조항 순위 : {e5_rank}위")
print(f"BGE 긴 조항 순위 : {bge_rank}위")

Token indices sequence length is longer than the specified maximum sequence length for this model (1714 > 512). Running this sequence through the model will result in indexing errors


E5 토큰 : 1714
BGE 토큰: 1714
E5  긴 조항 순위 : 12위
BGE 긴 조항 순위 : 1위


In [13]:
# 2) E5 prefix 누락 결과
prefix_df = pd.DataFrame([
    ["E5 정상 prefix", e5_scores[1]],
    ["E5 prefix 없음", e5_no_prefix_scores[1]],
], columns=["조건", "Hit@3"])

prefix_df["변화"] = prefix_df["Hit@3"] - e5_scores[1]
display(
    prefix_df.assign(
        **{
            "Hit@3": prefix_df["Hit@3"].map(lambda x: f"{x:.1%}"),
            "변화": prefix_df["변화"].map(lambda x: f"{x:+.1%}"),
        }
    )
)

,조건,Hit@3,변화
0,E5 정상 prefix,100.0%,+0.0%
1,E5 prefix 없음,100.0%,+0.0%


In [14]:
# 3) Normalization 누락
from sentence_transformers import SentenceTransformer

full_model = SentenceTransformer("BAAI/bge-m3", device="cpu")
raw_model = SentenceTransformer(
    modules=[full_model[0], full_model[1]],
    device="cpu",
)

raw_vecs = raw_model.encode(
    chunk_texts,
    normalize_embeddings=False,
)
raw_q = raw_model.encode(
    "연차를 다음 해로 넘길 수 있나요?",
    normalize_embeddings=False,
)

norm_vecs = raw_vecs / np.linalg.norm(
    raw_vecs, axis=1, keepdims=True
)
norm_q = raw_q / np.linalg.norm(raw_q)

raw_top = list(np.argsort(raw_vecs @ raw_q)[::-1][:5])
norm_top = list(np.argsort(norm_vecs @ norm_q)[::-1][:5])

print("정규화 안 함:", raw_top)
print("정규화 함  :", norm_top)
print("top5 동일 여부:", raw_top == norm_top)

정규화 안 함: [np.int64(6), np.int64(5), np.int64(7), np.int64(8), np.int64(3)]
정규화 함  : [np.int64(6), np.int64(5), np.int64(7), np.int64(8), np.int64(3)]
top5 동일 여부: True


## 최종 결과 정리

| 실험 | 검색 방식 | Hit@3 | Baseline 대비 |
|---|---|---:|---:|
| Baseline | BGE-M3 Dense | 73.3% | - |
| Track 2-1 | Dense + BM25 + RRF(60) | 100.0% | +26.7% |
| Track 2-2 | RRF k 조정 | 100.0% | +26.7% |
